# Stream Pipelines

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Build a stream pipeline: a source, steps such as filter, map, sorted and limit, then one terminal operation.
- Compute with number streams (sum, average, max) and handle an empty result with Optional.

**In this notebook**, one idea at a time (run each cell before moving on):
1. A stream pipeline
2. map, sorted and toList
3. Number streams and ranges
4. Terminal operations: getting an answer
5. Streams are lazy and single-use
6. Loops or streams?

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. A stream pipeline

A **stream** processes a collection as a pipeline: a **source** (`list.stream()`), any number of **intermediate steps** that each give a new stream, and **one terminal operation** that produces the result.

- `filter(test)` keeps the items that pass the test;
- `map(f)` turns each item into another;
- `sorted()` puts them in order;
- `sum()`, `count()`, `toList()`… end the pipeline.

The slides follow `[3, 8, 5, 12]` through `filter`, `map` and `sum`.

#### 🎞️ A stream pipeline, stage by stage

![Stream.of(3, 8, 5, 12): filter keeps values over 4, map doubles them to 16, 10, 24, and sum gives 50](stream-steps.svg)

In [ ]:
List<Integer> values = List.of(3, 8, 5, 12);
int result = values.stream()
        .filter(x -> x > 4)
        .mapToInt(x -> x * 2)
        .sum();
IO.println(result);
IO.println(values);   // the source list didn't change

🧪 **Try it:** remove the `filter` line and predict the sum first. Then keep only the even values instead of those over 4.

### ✏️ Exercise: Count the long words

Count the words with at least `min` letters, with a stream: filter, then count.

In [ ]:
long countLong(List<String> words, int min) {
    // TODO: words.stream().filter(...).count()
    return 0;
}

void main() {
    check("two long", countLong(List.of("a", "stream", "of", "words"), 4), 2L);
    check("none", countLong(List.of("a", "b"), 3), 0L);
}

<details>
<summary>Show an answer</summary>

```java
long countLong(List<String> words, int min) {
    return words.stream().filter(w -> w.length() >= min).count();
}

void main() {
    check("two long", countLong(List.of("a", "stream", "of", "words"), 4), 2L);
    check("none", countLong(List.of("a", "b"), 3), 0L);
}
```

</details>

## 2. map, sorted and toList

`map` turns each item into another, even of another type (a word into its length). `sorted()` uses the natural order, or a comparator you pass. `toList()` collects the result into a new list. `distinct()` drops repeats, and `limit(n)` keeps the first n.

In [ ]:
List<Integer> readings = List.of(12, -3, 7, 3, 0, -12);
List<Integer> sizes = readings.stream()
        .map(Math::abs)          // the size of each reading, without its sign
        .distinct()
        .sorted()
        .toList();
IO.println(sizes);
List<String> labels = sizes.stream().map(n -> "#" + n).toList();
IO.println(labels);

🧪 **Try it:** move `distinct()` before the `map` line: why do 3 and 12 now appear twice? Then add `.limit(2)`.

### ✏️ Exercise: Shout every word

Return a list with every word in upper case, with a stream: map, then toList.

In [ ]:
List<String> shout(List<String> words) {
    // TODO: map each word with w -> w.toUpperCase(Locale.ROOT)
    return List.of();
}

void main() {
    check("two words", shout(List.of("hi", "there")), List.of("HI", "THERE"));
    check("empty", shout(List.of()), List.of());
}

<details>
<summary>Show an answer</summary>

```java
List<String> shout(List<String> words) {
    return words.stream().map(w -> w.toUpperCase(Locale.ROOT)).toList();   // ROOT: the same in every country
}

void main() {
    check("two words", shout(List.of("hi", "there")), List.of("HI", "THERE"));
    check("empty", shout(List.of()), List.of());
}
```

</details>

## 3. Number streams and ranges

`mapToInt` and `mapToDouble` turn a stream of objects into a stream of numbers, which adds `sum`, `average`, `min` and `max`. `IntStream.range(a, b)` gives the numbers from `a` up to, but not including, `b` (`rangeClosed` includes `b`): a loop counter as a stream.

In [ ]:
List<String> words = List.of("stream", "map", "filter");
int letters = words.stream().mapToInt(String::length).sum();
IO.println(letters + " letters");
IO.println(IntStream.range(0, 5).boxed().toList());
IO.println(IntStream.rangeClosed(1, 10).filter(n -> n % 3 == 0).sum());
IntSummaryStatistics stats = IntStream.of(4, 8, 15, 16, 23, 42).summaryStatistics();
IO.println(stats.getMin() + " to " + stats.getMax() + ", average " + stats.getAverage());

🧪 **Try it:** compute 1 + 2 + … + 100 with `IntStream.rangeClosed`. Then the product of 1 to 5 with `.reduce(1, (a, b) -> a * b)`.

## 4. Terminal operations: getting an answer

The terminal operation decides what comes out:

| Operation | Gives |
|---|---|
| `count()` | how many items (a `long`) |
| `sum()`, `average()`, `max()` | on number streams (`mapToInt`, `mapToDouble`) |
| `anyMatch(test)`, `allMatch(test)` | `true` or `false` |
| `toList()` | a list |
| `collect(Collectors.joining(", "))` | one String |

`average()` and `max()` give an **Optional**, because an empty stream has no average: `orElse(0)` gives a fallback.

In [ ]:
List<Double> totals = List.of(120.0, 35.5, 60.0, 15.0);
IO.println(totals.stream().filter(t -> t >= 50).count() + " big orders");
IO.println("Average: " + totals.stream().mapToDouble(t -> t).average().orElse(0));
IO.println("Any over 100? " + totals.stream().anyMatch(t -> t > 100));
IO.println("All over 10? " + totals.stream().allMatch(t -> t > 10));
IO.println(totals.stream().map(t -> "$" + t).collect(Collectors.joining(", ")));

🧪 **Try it:** compute the average of an empty list (`List.<Double>of()`) with and without `orElse(0)`. Then find the largest total with `max()`.

## 5. Streams are lazy and single-use

Two surprises. Intermediate steps do **nothing** until a terminal operation runs: a pipeline without one never runs. And a stream can be used **only once**: to ask a second question, start again with a new `.stream()`.

In [ ]:
List<Integer> values = List.of(1, 2, 3);
values.stream().map(v -> {
    IO.println("mapping " + v);      // never printed: no terminal operation
    return v * 2;
});

Stream<Integer> once = values.stream().filter(v -> v > 1);
IO.println(once.count());
try {
    IO.println(once.count());        // the second use
} catch (IllegalStateException e) {
    IO.println("Error: " + e.getMessage());
}

🧪 **Try it:** add `.toList()` after the first `map` and run: now the "mapping" lines appear.

## 6. Loops or streams?

Streams shine when the job reads as a pipeline: keep some, transform them, collect or add up. A loop is clearer when steps depend on each other, when you need to stop early in a complicated way, or when you change things outside as you go. Common mistakes with streams:

- **Changing the source list** inside a stream: build a new list with `toList()` instead.
- **`Optional.get()` on an empty result:** use `orElse` or `ifPresent`.
- **Forgetting the terminal operation:** the pipeline never runs (step 5).

In [ ]:
List<Integer> values = List.of(3, 9, 4);
Optional<Integer> big = values.stream().filter(v -> v > 100).findFirst();
IO.println(big.isPresent() + " " + big.orElse(-1));
big.ifPresent(v -> IO.println("found " + v));     // prints nothing

🧪 **Try it:** call `big.get()` and read the exception. Then change the filter to `v > 5`.

## In short

A **stream** runs a pipeline over a collection: a source (`list.stream()`), any number of intermediate steps such as `filter`, `map`, `sorted` and `limit`, then **one terminal operation** that produces the result.

```java
void main() {
    List<Double> orders = List.of(120.0, 35.5, 60.0, 15.0, 80.0);
    List<Double> top = orders.stream()
            .filter(t -> t >= 50)
            .sorted(Comparator.reverseOrder())
            .limit(2)
            .toList();
    IO.println(top);                                   // [120.0, 80.0]
    double total = orders.stream().mapToDouble(t -> t).sum();
    OptionalDouble avg = orders.stream().mapToDouble(t -> t).average();
    IO.println(total + " " + avg.orElse(0));
    IO.println(IntStream.rangeClosed(1, 5).map(n -> n * n).sum());   // 55
}
```

Number streams (`mapToInt`, `mapToDouble`, `IntStream.range`) add `sum`, `average` and `max`; `average` and `max` give an Optional, because an empty stream has none.

**Watch out:** nothing runs until the terminal operation, and a stream can be used only once.

## Readings

- [Processing Data in Memory Using the Stream API](https://dev.java/learn/api/collections-and-streams/streams/map-filter-reduce/) (dev.java (Oracle), 9 min): The map-filter-reduce idea, then how a stream pipeline expresses it: mapping, filtering, and reducing to one result.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Even Squares** · Easy
- **Stream Statistics** · Easy
- **Clean Up a Name List** · Medium
- **Top 3 Scorers** · Hard